# Download biogeochemical (BGC) and physical (PHYS) analysis products from CMEMS

**Last updated: 21/10/2024**

## Import libraries and define paths

In [1]:
import os 
from pathlib import Path
import copernicusmarine
from datetime import datetime

In [2]:
# Path to download directory
PATH_ROOT_DIR = Path.cwd().resolve().parents[0] # /absolute/path/to/two/levels/up
full_path_download_dir = os.path.join(PATH_ROOT_DIR,"data","raw","CMEMS")
os.makedirs(full_path_download_dir, exist_ok=True)

## Set the download parameters

In [3]:
# =====================================================
# List of datasets
# =====================================================

LIST_DATASET_IDS = [

# Global Ocean Colour (Copernicus-GlobColour), Bio-Geo-Chemical, L4 (monthly and interpolated) from Satellite Observations (1997-ongoing), 4 x 4 km, monthly (1997-2024)
# Product ID: OCEANCOLOUR_GLO_BGC_L4_MY_009_104
    "cmems_obs-oc_glo_bgc-plankton_my_l4-multi-4km_P1M",        # Chlorophyll (mg m-3)
    "cmems_obs-oc_glo_bgc-pp_my_l4-multi-4km_P1M",              # NPP (mg C m-2 d-1)
    "cmems_obs-oc_glo_bgc-transp_my_l4-multi-4km_P1M",          # kd (m-1)
    
# Global Ocean Physics Reanalysis, 0.083° × 0.083°, monthly climatology (1993-2016)
# Product ID: GLOBAL_MULTIYEAR_PHY_001_030
    "cmems_mod_glo_phy_my_0.083deg-climatology_P1M-m"           # MLD (m) (defined using sigma theta), sea ice fraction, temperature
]

# ===============================================================================
# List of output file names (should correspond to the dataset names listed above)
# ===============================================================================

LIST_OUTPUT_NAMES = [
    "mod_bgc_glo_chla",
    "mod_bgc_glo_npp",
    "mod_bgc_glo_kd",
    "mod_phys_glo_mld",
    "mod_phys_glo_icefrac",
    "mod_phys_glo_temp"
]

# ===============================================================================
# List of variable names to download (search for small and large caps)
# ===============================================================================

LIST_VARIABLES = [
    "PP",
    "CHL",
    "KD490",
    "mlotst",
    "siconc",
    "thetao"
]

## Exploratory analysis of one dataset to get variable names

In [4]:
help(copernicusmarine.subset)

Help on function subset in module copernicusmarine.python_interface.subset:

subset(dataset_url: Optional[str] = None, dataset_id: Optional[str] = None, dataset_version: Optional[str] = None, dataset_part: Optional[str] = None, username: Optional[str] = None, password: Optional[str] = None, variables: Optional[List[str]] = None, minimum_longitude: Optional[float] = None, maximum_longitude: Optional[float] = None, minimum_latitude: Optional[float] = None, maximum_latitude: Optional[float] = None, minimum_depth: Optional[float] = None, maximum_depth: Optional[float] = None, vertical_dimension_as_originally_produced: bool = True, start_datetime: Union[datetime.datetime, str, NoneType] = None, end_datetime: Union[datetime.datetime, str, NoneType] = None, subset_method: Literal['nearest', 'strict'] = 'nearest', output_filename: Optional[str] = None, file_format: Literal['netcdf', 'zarr'] = 'netcdf', service: Optional[str] = None, request_file: Union[pathlib.Path, str, NoneType] = None, outp

In [5]:
DS = copernicusmarine.open_dataset(
    dataset_id = "cmems_obs-oc_glo_bgc-transp_my_l4-multi-4km_P1M"
)
DS

Fetching catalog: 100%|█████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:09<00:00,  3.17s/it]
INFO - 2024-10-26T11:22:03Z - Dataset version was not specified, the latest one was selected: "202311"
INFO - 2024-10-26T11:22:03Z - Dataset part was not specified, the first one was selected: "default"
INFO - 2024-10-26T11:22:05Z - Service was not specified, the default one was selected: "arco-geo-series"


<xarray.Dataset> Size: 449GB
Dimensions:            (time: 325, latitude: 4320, longitude: 8640)
Coordinates:
  * latitude           (latitude) float32 17kB -89.98 -89.94 ... 89.94 89.98
  * longitude          (longitude) float32 35kB -180.0 -179.9 ... 179.9 180.0
  * time               (time) datetime64[ns] 3kB 1997-09-01 ... 2024-09-01
Data variables:
    KD490              (time, latitude, longitude) float32 49GB ...
    KD490_uncertainty  (time, latitude, longitude) float64 97GB ...
    SPM                (time, latitude, longitude) float32 49GB ...
    SPM_uncertainty    (time, latitude, longitude) float64 97GB ...
    ZSD                (time, latitude, longitude) float32 49GB ...
    ZSD_uncertainty    (time, latitude, longitude) float64 97GB ...
    flags              (time, latitude, longitude) int8 12GB ...
Attributes: (12/91)
    Conventions:                     CF-1.8, ACDD-1.3
    DPM_reference:                   GC-UD-ACRI-PUG
    IODD_reference:                  GC-UD-ACRI-PUG
    acknowledgement:                 The Licensees will ensure that original ...
    citation:                        The Licensees will ensure that original ...
    cmems_product_id:                OCEANCOLOUR_GLO_BGC_L4_MY_009_104
    ...                              ...
    time_coverage_end:               2023-11-01T05:36:00Z
    time_coverage_resolution:        P31D
    time_coverage_start:             2023-09-30T19:41:10Z
    title:                           cmems_obs-oc_glo_bgc-transp_my_l4-multi-...
    westernmost_longitude:           -180.0
    westernmost_valid_longitude:     -180.0

## Download data

In [7]:
%%time

for dataset_id in LIST_DATASET_IDS:

    DS = copernicusmarine.open_dataset(dataset_id = dataset_id)
    
    for variable_name, output_name in zip(LIST_VARIABLES, LIST_OUTPUT_NAMES):
        
        if variable_name in DS.data_vars: # if the variable exists, do something with it

            print(f"Downloading {variable_name} from {dataset_id}")
            copernicusmarine.subset(
                dataset_id=dataset_id,
                variables={variable_name},
                minimum_longitude=-180,
                maximum_longitude=180,
                minimum_latitude=-90,
                maximum_latitude=90,
                output_filename=f"{output_name}.nc",
                output_directory=f"{full_path_download_dir}",
                force_download=True,
            )

print('All datasets downloaded!')

INFO - 2024-10-26T11:23:54Z - Dataset version was not specified, the latest one was selected: "202311"
INFO - 2024-10-26T11:23:54Z - Dataset part was not specified, the first one was selected: "default"
INFO - 2024-10-26T11:23:55Z - Service was not specified, the default one was selected: "arco-geo-series"


INFO - 2024-10-26T11:23:58Z - Dataset version was not specified, the latest one was selected: "202311"
INFO - 2024-10-26T11:23:58Z - Dataset part was not specified, the first one was selected: "default"
INFO - 2024-10-26T11:23:59Z - Service was not specified, the default one was selected: "arco-geo-series"
WARNING - 2024-10-26T11:23:59Z - Some or all of your subset selection [-90, 90] for the latitude dimension  exceed the dataset coordinates [-89.97917175292969, 89.97916412353516]
WARNING - 2024-10-26T11:23:59Z - Some or all of your subset selection [-180.0, 180.0] for the longitude dimension  exceed the dataset coordinates [-179.9791717529297, 179.9791717529297]
INFO - 2024-10-26T11:23:59Z - Downloading using service arco-geo-series...
INFO - 2024-10-26T11:24:00Z - Estimated size of the dataset file is 46299.847 MB.
INFO - 2024-10-26T11:24:00Z - Writing to local storage. Please wait...


  0%|          | 0/52652 [00:00<?, ?it/s]

INFO - 2024-10-26T11:47:27Z - Successfully downloaded to /Users/Anna/LocalDocuments/Academic/Projects/ocean-data-lab/data/raw/CMEMS/mod_bgc_glo_kd.nc


All datasets downloaded!
CPU times: user 4min 31s, sys: 17min 54s, total: 22min 26s
Wall time: 23min 33s
